Plan:

create a bigram model to make new words from the list of 32k names

1. Load the data set names.txt
2. Iterate over each word and then create pair of 2 characters each.
3. add a Special Symbol `S` and `E` for begining and ending.
4. add them to a count like dictionary --> convert them to tensors when we do so we use the ch1 as the first column and the ch2 as the second one and basically its gonna tell how often the two occur together.

In [4]:
!pip install torch

In [5]:
words = open("/content/names.txt", 'r').read().splitlines()

FileNotFoundError: [Errno 2] No such file or directory: '/content/names.txt'

In [ ]:
b = {}
for w in words:
  chs = ['<S>'] + list(w) + ['<E>']
  for ch1, ch2 in zip(chs, chs[1:]):
    bigram = (ch1, ch2)
    b[bigram] = b.get(bigram, 0) + 1

In [ ]:
## How this worked: b.items -> [(ch1, ch2), count] then key decides on what to sort -> kv is the tuple and the second item i.e. the count of the bigram to sort on '-' sign just makes it decending.
## This is basically for analysis of the data
sorted(b.items(), key = lambda kv: -kv[1])

From here onwards we work to generate the tensors

- use the set of characters in the data set to do so we create a string of all the data set and pass it through set builders to remove duplicates and generate a clean sorted set of **charcters used in out DB**
- Now Tensors that we create are basically number based float or int so we create a mapping / lookup / base embedding for the model. To do so we create the `stoi` string to integer mapping using the enumerate function

In [ ]:
import torch

N = torch.zeros((28,28), dtype = torch.int32) # why 28/28 array is we have 26 letters a-z and then two special chars for start and end

chars = sorted(list(set(''.join(words))))
stoi = {s:i for i,s in enumerate(chars)}
stoi["<S>"] = 26
stoi["<E>"] = 27


In [ ]:
for w in words:
  chs = ["<S>"] + list(w) + ["<E>"]
  for ch1,ch2 in zip(chs, chs[1:]):
    idx1 = stoi[ch1]
    idx2 = stoi[ch2]
    N[idx1, idx2] += 1 # basically here we are adding the count say 'a' appeared after '<S>' so we pluck row of idx[a] column of idx[<S>] and check and add 1 to it



In [ ]:
N

In [ ]:
itos = {i:s for s,i in stoi.items()}

import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(28):
  for j in range(28):
    chstr = itos[i] + itos[j]
    plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
    plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off')

In [ ]:
## Now in this case we observe that <E> at begining never happens so the row is 0 and word never ends with <S> so the column of <S> is empty.
## We need to remove it also instead of 2 start and end symbols we will have 1 symbol that is '.'
## Compared to the results above and here we can see that the emply blocks are cleared

import torch

N = torch.zeros((27,27), dtype = torch.int32) # why 27/27 array is we have 26 letters a-z and then only special character '.'

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi["."] = 0

for w in words:
  chs = ["."] + list(w) + ["."]
  for ch1,ch2 in zip(chs, chs[1:]):
    idx1 = stoi[ch1]
    idx2 = stoi[ch2]
    N[idx1, idx2] += 1 # basically here we are adding the count say 'a' appeared after '<S>' so we pluck row of idx[a] column of idx[<S>] and check and add 1 to it


itos = {i:s for s,i in stoi.items()}

import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(27):
  for j in range(27):
    chstr = itos[i] + itos[j]
    plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
    plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off')


In [ ]:
## In this part we need to understand the prequal of this multinomial
## torch.multinomial is an important function --> basically the working is as follows.
## it takes a probablity distribution and based on it generates num_samples = n over that distribution.

g = torch.Generator().manual_seed(2147483647)
p = torch.rand(3, generator=g)
p = p / p.sum()
print(p)
torch.multinomial(p, num_samples=10, replacement=True, generator=g)

In [ ]:
## Now we work on the probablity distributions
p = N[0].float() ## We convert to float since its gonna be decimals
p = p / p.sum()
p

In [ ]:
## What we did here?
### We used the distribution of our intended row and sampled a letter from it, then we use the row of that letter and choose the next till we go to end of word token that is .
### To implement this idea we use a loop

g = torch.Generator().manual_seed(2147483647)
ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
itos[ix]

In [ ]:
ix = 0 #we start with an initial 0 index
g = torch.Generator().manual_seed(2147483647)
for i in range(10):
  out = []
  while True:
    p = N[ix].float() # take the row
    p = p / p.sum() # normalize
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item() # pick a word
    out.append(itos[ix]) # append the word
    if ix == 0: # if the word says to stop we stop
      break
  print("".join(out))


In the below operation we use the sum in.a different way
We want to divide each row of the matrix N which is saved as P_Matrix below as the copy of N in float format by the rows sum

we use matrix.sum(axis = 0 column sum/1 row sum) keepdims true ==> this ensures that the matrix keeps the row and not just sum it and throw out.

basically keep dim true create a matrix 27,1 which is 27 rows for 27 rows in N and 1 column as the sum of each row.

if we do axis 0 it will create a 1,27 matrix that is sumof columns

what happens if we set the keep_dims to false --> we get 27, _ matrix there is no dims

BRODCASTABLE true / false means is the operation doable for that either the counts of rows and columns has to be same or either is 1 or either does not exist.



In [ ]:
## We do a bit of efficieny : Since we are calculating the normalization for each row innthe while loop it gets rework instead we create a Probablity matrix

P_Matrix = N.float() # Copy of the N in float form
P_Matrix = P_Matrix / P_Matrix.sum(1, keepdims=True) # Normalize the rows
